# Data mining \& clustering

The goal if this practical is to adress the folowing problem:
<center style="color:red" >  Given XXX raw, unlabeled documents, ... How to exploit/understand/represent them?</center>

In the previous week, we have seen how to represent textual data with the Bag of Words (BoW) model:
$$X =
	\begin{matrix}
	 & \textbf{t}_j \\
	 & \downarrow \\
	\textbf{d}_i \rightarrow &
	\begin{pmatrix}
	x_{1,1} & \dots & x_{1,d} \\
	\vdots & \ddots & \vdots \\
	x_{N,1} & \dots & x_{N,d} \\
	\end{pmatrix}
	\end{matrix}
	$$

From this BoW representation, we want to answer the following questions:
1. Which clustering algorithm to choose?
    - K-means, LSA, pLSA, LDA
1. What results to expect?
    - Semantics, noise cleaning, etc...
1. Which qualitative and quantitative analyses to understand the groups?
[comment]: <> (%1. Comment boucler, itérer pour améliorer la qualité du processus?)


<span style="color:magenta" > In this practical, we use a **labeled dataset** in order to evaluate performances with quantitative and well-defined metrics. </span>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import codecs
import re
import os.path
import sklearn

## Data loading



In [ ]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')

In [ ]:
# conversion BoW + tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer
#vectorizer = TfidfVectorizer()
vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=1000, stop_words='english')

vectors = vectorizer.fit_transform(newsgroups_train.data)
print(vectors.shape)

# sparsity measure = 44 active words over 1000 per document (157 over 130000) !!
print(vectors.nnz / float(vectors.shape[0]))

In [ ]:
# retrieve words
print([(i,vectorizer.get_feature_names_out()[i]) \
       for i in np.random.randint(vectors.shape[1], size=10)])

In [ ]:
# labels (only for evaluation)
Y = newsgroups_train.target
print(Y[:10])
print([newsgroups_train.target_names[i] for i in Y[:20]]) # vraie classe

# 0) Word clouds
### Drawing word clouds from the raw corpus or words' frequencies :  [make word clouds !](https://github.com/amueller/word_cloud)

### Installation
If you are using pip:

`pip install wordcloud`

### If you are using conda, you can install from the conda-forge channel:

`conda install -c conda-forge wordcloud`

### Let's look at the most frequent words in this dataset

In [ ]:

from collections import Counter
data = np.array(newsgroups_train.data)
corpus = "".join(data)
words = corpus.split() 
print("Nb mots=",len(words))
from wordcloud import WordCloud, STOPWORDS



word_counts = Counter(words)
most_common = word_counts.most_common(30)
for word, count in most_common:
    print(f"{word}: {count}")

### Plot the N frequent words and verify that its follows a Zipf law

In [ ]:
from collections import Counter

word_freq = Counter(words)
print(word_freq.most_common(10))

n=50
_, freq_words = zip(*word_freq.most_common(n))

plt.figure(figsize=(10,8))
plt.bar(np.linspace(1,n+1), freq_words)

plt.xlabel("Words")
plt.ylabel("Frequency")
plt.title("Frequency of words")
plt.show()here

### Experiment word clouds

In [ ]:
wordcloud = WordCloud(background_color='white', stopwords = [], max_words=100).generate(corpus)

plt.figure()
plt.imshow(wordcloud)
plt.axis('off')

In [ ]:
from wordcloud import STOPWORDS # Note: this is the default option
wordcloud = WordCloud(background_color='white', stopwords = STOPWORDS, max_words=100).generate(corpus)

plt.figure()
plt.imshow(wordcloud)
plt.axis("off")

### Use word clouds with generate\_from\_frequencies.
N.B.: retrieve the most words frequencies using a CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer(stop_words='english', max_features=500)
count_matrix = count_vec.fit_transform(newsgroups_train.data)
freqs = dict(zip(count_vec.get_feature_names_out(),
                  count_matrix.sum(axis=0).A1))

wordcloud = WordCloud(background_color='white', max_words=100).generate_from_frequencies(freqs)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud)
plt.axis('off')
plt.title('Word Cloud from CountVectorizer frequencies')
plt.show()

### Drawing word clouds from classes


In [ ]:
# Word clouds for a selection of classes
selected_classes = [0, 4, 10, 15]  # pick 4 classes
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, cls in enumerate(selected_classes):
    ax = axes[idx // 2][idx % 2]
    class_docs = [newsgroups_train.data[i] for i in range(len(Y)) if Y[i] == cls]
    class_text = ' '.join(class_docs)
    wc = WordCloud(background_color='white', stopwords=STOPWORDS,
                   max_words=80).generate(class_text)
    ax.imshow(wc)
    ax.axis('off')
    ax.set_title(newsgroups_train.target_names[cls], fontsize=14)

plt.suptitle('Word Clouds by class', fontsize=16)
plt.tight_layout()
plt.show()

# 1) Clustering algorithm: K-Means

**Let's start by the most famous and simple unsupervised algorithm: $k$-means!**
Look at [sklear documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
and apply it to your BoW matrix.


In [ ]:
from sklearn.cluster import KMeans
# your code here
kmeans = KMeans(n_clusters=20, random_state=0, max_iter=10).fit(vectors)
# Getting clusters:
kmeans.cluster_centers_


### Clustering Analysis:
1. **Qualtitative:**
    - Look at the most important words for each cluster
    - Perform cluster assignement to each document, and compute word cloud on the document (raw text or frequencies)
2. **Quantitative:**
    - Compute cluster "purity": $p_j= |y^*_j|$, where $y^*_j$ is the most frequent (GT) label in cluster $C_j$ $\Rightarrow$ $p = \frac{1}{N}\sum\limits_j  p_j$
    - Compute [Rand Score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.rand_score.html) and [Adjusted Rand Score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html#sklearn.metrics.adjusted_rand_score)

In [ ]:
# Qualitative: top words per cluster
feature_names = vectorizer.get_feature_names_out()
n_top_words = 10

print("=" * 60)
print("K-MEANS: Top words per cluster")
print("=" * 60)
for i in range(20):
    top_indices = kmeans.cluster_centers_[i].argsort()[::-1][:n_top_words]
    top_words = [feature_names[j] for j in top_indices]
    print(f"Cluster {i:2d}: {', '.join(top_words)}")

In [ ]:
# Word clouds per K-Means cluster
fig, axes = plt.subplots(4, 5, figsize=(20, 16))

for i in range(20):
    ax = axes[i // 5][i % 5]
    cluster_freqs = dict(zip(feature_names, kmeans.cluster_centers_[i]))
    wc = WordCloud(background_color='white', max_words=50).generate_from_frequencies(cluster_freqs)
    ax.imshow(wc)
    ax.axis('off')
    ax.set_title(f'Cluster {i}', fontsize=10)

plt.suptitle('K-Means: Word Clouds per cluster', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative evaluation
from sklearn.metrics import rand_score, adjusted_rand_score

def compute_purity(y_true, y_pred):
    """Compute cluster purity."""
    contingency = np.zeros((y_pred.max() + 1, y_true.max() + 1), dtype=int)
    for yt, yp in zip(y_true, y_pred):
        contingency[yp, yt] += 1
    return contingency.max(axis=1).sum() / len(y_true)

purity_km = compute_purity(Y, kmeans.labels_)
rs_km = rand_score(Y, kmeans.labels_)
ars_km = adjusted_rand_score(Y, kmeans.labels_)

print("=" * 40)
print("K-MEANS EVALUATION")
print("=" * 40)
print(f"Purity              : {purity_km:.4f}")
print(f"Rand Score           : {rs_km:.4f}")
print(f"Adjusted Rand Score  : {ars_km:.4f}")

# 2) Latent Semantic Analysis (LSA <=> SVD)


**Remember the LSA factorziation**:
$$
\begin{matrix}
 & X  &\!\!\!\!\!=\!\!\!\!\!& U  & \Sigma & V^T \\
  & \textbf{t}_j   &  & \hat{ \textbf{d}_i} & &  \\
 & \downarrow  &  &\downarrow  & & \\
\textbf{d}_i \rightarrow
&
\begin{pmatrix}
x_{1,1} & \dots & x_{1,d} \\
\\
\vdots & \ddots & \vdots \\
\\
x_{N,1} & \dots & x_{N,d} \\
\end{pmatrix}
&
\!\!\!\!\!=\!\!\!\!\!
%&
%(\hat{ \textbf{t}_j}) \rightarrow
&
\begin{pmatrix}
\begin{pmatrix} &  \textbf{u}_1 &  \end{pmatrix} \\
\vdots \\
\begin{pmatrix}  & \textbf{u}_k &  \end{pmatrix}
\end{pmatrix}
%&
%\!\!\!\!\!\cdot\!\!\!\!\!
&
\begin{pmatrix}
\sigma_1 & \dots & 0 \\
\vdots & \ddots & \vdots \\
0 & \dots & \sigma_k \\
\end{pmatrix}
%&
%\!\!\!\!\!\cdot\!\!\!\!\!
&
\begin{pmatrix}
\begin{pmatrix} \, \\ \, \\ \textbf{v}_1 \\ \, \\ \,\end{pmatrix}
\dots
\begin{pmatrix} \, \\ \, \\ \textbf{v}_k \\ \, \\ \, \end{pmatrix}
\end{pmatrix}
\end{matrix}
$$

- Look at [SVD doc in skelarn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html#sklearn.decomposition.TruncatedSVD)
- Do the same qualitative/quantitative evaluation than with K-Means
- You can also use LSA as a pre-processing step for K-Means, *i.e.* running K-Means on $\boldsymbol{U}$ matrix above
    - N.B. : try without/with $\ell_2$ normalization of $\boldsymbol{U}$'s rows before running  K-Means
    - You can also benefit from LSA pre-processing for using [t-SNE visualization](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) (see code below)


In [ ]:
from matplotlib import cm
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.manifold import TSNE




n_components = 20
svd = TruncatedSVD(n_components=n_components, random_state=0)
U = svd.fit_transform(vectors)
U_norm = Normalizer(copy=False).fit_transform(U)

kmeans_lsa_norm = KMeans(n_clusters=20, random_state=0, max_iter=10).fit(U_norm)
labels_lsa_norm = kmeans_lsa_norm.labels_

print(f"LSA: {n_components} components, explained variance {svd.explained_variance_ratio_.sum():.3f}")
print(f"Pureté: {compute_purity(Y, labels_lsa_norm):.4f}")
print(f"Rand Score: {rand_score(Y, labels_lsa_norm):.4f}")
print(f"Adjusted Rand Score: {adjusted_rand_score(Y, labels_lsa_norm):.4f}")

tsne = TSNE(n_components=2, init='pca', max_iter=5000, verbose=2, random_state=0)
tsne_mat = tsne.fit_transform(U_norm)

NN2cluster = np.argmax(np.abs(U_norm), axis=0)  # one representative document per LSA latent dimension

cmap = cm.get_cmap('tab20', 20)
plt.figure(figsize=(15, 10))
plt.scatter(tsne_mat[:, 0], tsne_mat[:, 1], c=Y, cmap=cmap, s=8, alpha=0.7)
plt.scatter(tsne_mat[NN2cluster, 0], tsne_mat[NN2cluster, 1],
            c='black', s=120, marker='X', label='LSA dimension representatives')
plt.title("t-SNE sur U_norm (LSA) avec pts de référence (NN2cluster-style)")
plt.colorbar(ticks=range(20))
plt.legend(loc='best')
plt.show()

# 3) Latent Dirichlet Allocation (LDA)

Perform the same experiments with LDA:
- LDA
https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html


**Start with a CountVectorizer**

In [ ]:
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import CountVectorizer

# Initialize regex tokenizer
tokenizer = RegexpTokenizer(r'\w+')

# Vectorize document using TF-IDF
vectorizer = CountVectorizer(lowercase=True,
                        stop_words='english',
                        ngram_range = (1,1),
                        tokenizer = tokenizer.tokenize, max_df=0.95, min_df=2, max_features=1000)

vectors = vectorizer.fit_transform(newsgroups_train.data)
print(vectors.shape)
print(vectors.nnz / float(vectors.shape[0]))



In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=20, random_state=0, max_iter=10)
lda.fit(vectors)


## LDA-viz

In [ ]:
import pyLDAvis.lda_model as pyLDAvis_sklearn
pyLDAvis.enable_notebook()
pyLDAvis_sklearn.prepare(lda, vectors, vectorizer)


# Performances evaluation

**Compare the different approaches wrt three quantitative metrics.**

In [ ]:
import pandas as pd
from sklearn.metrics import rand_score, adjusted_rand_score

# K-Means sur TF-IDF (re-fit sur vectors TF-IDF si nécessaire)
km_labels = kmeans.labels_

# LSA argmax (assigner chaque doc à sa composante dominante)
lsa_labels = np.argmax(np.abs(U), axis=1)

# K-Means sur LSA sans norm
km_lsa_labels = KMeans(n_clusters=20, random_state=0, max_iter=10).fit(U).labels_

# K-Means sur LSA avec norm L2
from sklearn.preprocessing import Normalizer
U_norm = Normalizer().fit_transform(U)
km_lsa_n_labels = KMeans(n_clusters=20, random_state=0, max_iter=10).fit(U_norm).labels_

# LDA
lda_labels = np.argmax(lda.transform(vectors), axis=1)

# Tableau
methods = {
    'K-Means (TF-IDF)': km_labels,
    'LSA (argmax)': lsa_labels,
    'K-Means on LSA (no norm)': km_lsa_labels,
    'K-Means on LSA (L2 norm)': km_lsa_n_labels,
    'LDA': lda_labels
}

rows = []
for name, labels in methods.items():
    rows.append({
        'Method': name,
        'Purity': compute_purity(Y, labels),
        'Rand Score': rand_score(Y, labels),
        'Adj. Rand Score': adjusted_rand_score(Y, labels)
    })

results = pd.DataFrame(rows)
print("=" * 70)
print("FINAL COMPARISON")
print("=" * 70)
print(results.to_string(index=False))


In [ ]:
# Bar plot comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
methods = results['Method']
x = np.arange(len(methods))

for idx, metric in enumerate(['Purity', 'Rand Score', 'Adj. Rand Score']):
    axes[idx].bar(x, results[metric], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3'])
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    axes[idx].set_title(metric, fontsize=13)
    axes[idx].set_ylim(0, 1)
    for i, v in enumerate(results[metric]):
        axes[idx].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Clustering Performance Comparison', fontsize=15)
plt.tight_layout()
plt.show()